# Praktikum Pertemuan 6 Data Science
Nama : Rangga Saputra

NIM : 250401020034

Dosen Pengajar : Syahid Abdullah, S.Si., M.Kom.

## Langkah 1: Load & EDA Singkat

Muat dataset, periksa missing values, tipe data, dan distribusi target untuk memahami kondisi data sebelum diproses.

In [119]:
import pandas as pd, seaborn as sns
import matplotlib.pyplot as plt

df = sns.load_dataset('titanic')

# Pilih kolom yang akan digunakan
cols = ['pclass','sex','age','sibsp','parch','fare','embarked','survived']
df = df[cols].copy()

print('Shape:', df.shape)
print('\nMissing values:')
print(df.isnull().sum())
print('\nDistribusi target:')
print(df['survived'].value_counts(normalize=True).round(3))

# survived=0: ~61.6%, survived=1: ~38.4% — kelas tidak seimbang!

Shape: (891, 8)

Missing values:
pclass        0
sex           0
age         177
sibsp         0
parch         0
fare          0
embarked      2
survived      0
dtype: int64

Distribusi target:
survived
0   0.616
1   0.384
Name: proportion, dtype: float64


In [120]:
# Preview data awal
df.head()

,pclass,sex,age,sibsp,parch,fare,embarked,survived
0,3,male,22.000,1,0,7.250,S,0
1,1,female,38.000,1,0,71.283,C,1
2,3,female,26.000,0,0,7.925,S,1
3,1,female,35.000,1,0,53.100,S,1
4,3,male,35.000,0,0,8.050,S,0


## Langkah 2: Handling Missing Values

Nilai yang hilang diisi sebelum encoding. Kolom numerik diisi dengan median karena lebih robust terhadap outlier, sedangkan kolom kategorikal diisi dengan modus (nilai yang paling sering muncul).

In [121]:
# Age: isi dengan median (robust terhadap outlier)
df['age'] = df['age'].fillna(df['age'].median())

# Embarked: isi dengan modus (nilai paling sering)
df['embarked'] = df['embarked'].fillna(df['embarked'].mode()[0])

print('Missing setelah handling:')
print(df.isnull().sum()) # Semua harus 0

Missing setelah handling:
pclass      0
sex         0
age         0
sibsp       0
parch       0
fare        0
embarked    0
survived    0
dtype: int64


In [122]:
# Preview setelah handling missing values
df.head()

,pclass,sex,age,sibsp,parch,fare,embarked,survived
0,3,male,22.000,1,0,7.250,S,0
1,1,female,38.000,1,0,71.283,C,1
2,3,female,26.000,0,0,7.925,S,1
3,1,female,35.000,1,0,53.100,S,1
4,3,male,35.000,0,0,8.050,S,0


## Langkah 3: Encoding Kategorikal

Kolom `sex` dan `embarked` bertipe string sehingga tidak bisa diproses model ML. One-Hot Encoding mengubahnya menjadi kolom biner (0/1). Parameter `drop_first=True` digunakan untuk menghindari dummy variable trap.

In [123]:
# One-Hot Encoding untuk 'sex' dan 'embarked'
df = pd.get_dummies(df,
                    columns=['sex', 'embarked'],
                    drop_first=True, # hindari dummy variable trap
                    dtype=int)

print('Kolom setelah encoding:')
print(df.columns.tolist())
# ['pclass','age','sibsp','parch','fare','survived',
#  'sex_male','embarked_Q','embarked_S']

Kolom setelah encoding:
['pclass', 'age', 'sibsp', 'parch', 'fare', 'survived', 'sex_male', 'embarked_Q', 'embarked_S']


In [124]:
# Preview setelah encoding — kolom sex dan embarked sudah jadi numerik
df.head()

,pclass,age,sibsp,parch,fare,survived,sex_male,embarked_Q,embarked_S
0,3,22.000,1,0,7.250,0,1,0,1
1,1,38.000,1,0,71.283,1,0,0,0
2,3,26.000,0,0,7.925,1,0,0,1
3,1,35.000,1,0,53.100,1,0,0,1
4,3,35.000,0,0,8.050,0,1,0,1


## Langkah 4: Train-Test Split

Dataset dibagi menjadi training set dan test set **sebelum scaling** untuk menghindari data leakage. Parameter `stratify=y` memastikan proporsi kelas `survived` tetap sama di kedua set.

In [125]:
from sklearn.model_selection import train_test_split

X = df.drop('survived', axis=1)
y = df['survived']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y # proporsi kelas terjaga
)

print(f'Train: {X_train.shape[0]} baris')
print(f'Test : {X_test.shape[0]} baris')
print('\nProporsi survived di Train:')
print(y_train.value_counts(normalize=True).round(3))
print('\nProporsi survived di Test:')
print(y_test.value_counts(normalize=True).round(3))

Train: 712 baris
Test : 179 baris

Proporsi survived di Train:
survived
0   0.617
1   0.383
Name: proportion, dtype: float64

Proporsi survived di Test:
survived
0   0.615
1   0.385
Name: proportion, dtype: float64


## Langkah 5: Feature Scaling

StandardScaler diterapkan hanya pada kolom numerik. Scaler di-`fit` hanya pada training set, lalu digunakan untuk `transform` test set bukan `fit_transform` agar statistik dari test set tidak bocor ke proses pelatihan.

In [126]:
from sklearn.preprocessing import StandardScaler

# Hanya kolom numerik yang perlu di-scale
# Kolom biner (sex_male, embarked_Q, embarked_S) TIDAK perlu
num_cols = ['pclass', 'age', 'sibsp', 'parch', 'fare']

scaler = StandardScaler()

# fit_transform pada training set (belajar μ dan σ dari sini)
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])

# transform saja pada test set (gunakan μ dan σ dari training!)
X_test[num_cols] = scaler.transform(X_test[num_cols])

print('Mean scaler (dari train):', scaler.mean_.round(2))
print('Std scaler (dari train):', scaler.scale_.round(2))
print()
print('Contoh X_train setelah scaling:')
print(X_train.head(3).round(3))
print('\nData siap dilatih model Machine Learning!')
print(f'X_train: {X_train.shape}, y_train: {y_train.shape}')
print(f'X_test : {X_test.shape}, y_test : {y_test.shape}')

Mean scaler (dari train): [ 2.31 29.46  0.49  0.39 31.82]
Std scaler (dari train): [ 0.83 13.03  1.06  0.84 48.03]

Contoh X_train setelah scaling:
     pclass    age  sibsp  parch   fare  sex_male  embarked_Q  embarked_S
692   0.830 -0.112 -0.465 -0.466  0.514         1           0           1
481  -0.371 -0.112 -0.465 -0.466 -0.663         1           0           1
527  -1.571 -0.112 -0.465 -0.466  3.955         1           0           1

Data siap dilatih model Machine Learning!
X_train: (712, 8), y_train: (712,)
X_test : (179, 8), y_test : (179,)
